## Setup

In [1]:
import os
import json
import random
import asyncio
from dataclasses import dataclass
from dotenv import load_dotenv
import aiohttp
from datasets import load_dataset
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm.asyncio import tqdm

from config import PROBE_QUESTIONS, REWRITER_MODEL, TARGET_MODEL, SCORER_MODEL, N_SAMPLES_PER_CLASS
from data.rewriter import rewrite_all


/root/cambria-final-project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
random.seed(0)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

In [3]:
@dataclass
class Sample:
    """A single sample for evaluation awareness testing."""
    id: str
    transcript: str
    is_eval: int  # 1 = evaluation, 0 = real
    source: str

## Dataset to json

Converts raw datasets to a json file for later access

In [4]:
@dataclass
class Sample:
    """A single sample for evaluation awareness testing."""
    id: str
    transcript: str
    is_eval: int  # 1 = evaluation, 0 = real
    source: str
    ans: str | None

In [5]:
def save_samples(dataset_name: str, samples: list[Sample] | list[dict], n_samples: int = N_SAMPLES_PER_CLASS, output_file: str = "data/prompt_data.json"):
    """Save dataset samples to a JSON file for documentation."""
    from datetime import datetime
    from dataclasses import asdict

    with open(output_file, 'r') as f:
        output = json.load(f)

    if isinstance(samples[0], dict):
        output[dataset_name] = samples
    else:
        output[dataset_name] = [asdict(s) for s in samples]
    output['metadata']['timestamp'] = datetime.now().isoformat()

    with open(output_file, "w") as f:
        json.dump(output, f, indent=2)

def load_mmlu_for_save(n: int = N_SAMPLES_PER_CLASS) -> list[Sample]:
    """Load MMLU questions as evaluation samples."""
    dataset = load_dataset("cais/mmlu", "all", split="test", trust_remote_code=True)

    random.seed(0)
    indices = random.sample(range(len(dataset)), min(n, len(dataset)))

    samples = []
    for idx in indices:
        item = dataset[idx]
        choices = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(item["choices"])])
        transcript = f"Question: {item['question']}\n\nChoices:\n{choices}"

        samples.append(Sample(
            id=f"mmlu_{idx}",
            transcript=transcript,
            is_eval=1,
            source="mmlu"
        ))

    return samples

def load_sharegpt_for_save(n: int = N_SAMPLES_PER_CLASS) -> list[Sample]:
    """Load ShareGPT conversations as real-world samples."""
    dataset = load_dataset(
        "anon8231489123/ShareGPT_Vicuna_unfiltered",
        data_files="ShareGPT_V3_unfiltered_cleaned_split_no_imsorry.json",
        split="train"
    )

    random.seed(0)
    indices = random.sample(range(len(dataset)), min(n*2, len(dataset)))

    samples = []
    for idx in indices:
        item = dataset[idx]

        conversations = item.get("conversations", [])
        if not conversations:
            continue

        first_human_msg = None
        for msg in conversations:
            if msg.get("from") == "human":
                first_human_msg = msg.get("value", "")
                break

        if not first_human_msg or len(first_human_msg.strip()) < 10:
            continue

        samples.append(Sample(
            id=f"sharegpt_{idx}",
            transcript=first_human_msg,
            is_eval=0,
            source="sharegpt"
        ))

        if len(samples) >= n:
            break

    return samples

def load_aime_for_save():
    aime1 = load_dataset('opencompass/AIME2025', 'AIME2025-I')['test']
    aime2 = load_dataset('opencompass/AIME2025', 'AIME2025-II')['test']

    questions = aime1['question'] + aime2['question']
    answers = aime1['answer'] + aime2['answer']

    samples = []
    for idx, (q, a) in enumerate(zip(questions, answers)):
        samples.append(Sample(
            id=f'aime_{idx}',
            transcript=q,
            is_eval=1,
            source='aime',
            ans=a,
        ))
    
    return samples


In [6]:
def rephrase_mmlu_for_save():
    with open('data/prompt_data.json', 'r') as f:
        mmlu_from_saved = json.load(f)['mmlu']
    
    return rewrite_all(mmlu_from_saved, question_type='mcq')

def rephrase_aime_for_save():
    with open('data/prompt_data.json', 'r') as f:
        aime_from_saved = json.load(f)['aime']
    
    return rewrite_all(aime_from_saved, question_type='math_v4')



In [7]:
# Run ONCE

# save_samples('mmlu', samples=load_mmlu_for_save())
# save_samples('sharegpt', samples=load_sharegpt_for_save())
# save_samples('mmlu_rephrased', samples=rephrase_mmlu_for_save())
# save_samples('aime', samples=load_aime_for_save())
save_samples('aime_rephrased_v4', samples=rephrase_aime_for_save())

100%|██████████| 30/30 [02:15<00:00,  4.53s/it]


## Benchmark Performance Evaluation

Measure how well the model actually performs on AIME problems (original vs rephrased).

In [ ]:
from config import SCORER_MODEL

# Canonical AIME prompt format used in benchmarks
# Reference: https://artificialanalysis.ai/evaluations/aime-2025

AIME_SYSTEM_PROMPT = """You are an expert mathematician. Solve problems step by step, showing your reasoning clearly.

AIME answers are always integers from 0 to 999.

Put your final answer inside \\boxed{} like this: \\boxed{123}"""


def format_aime_problem(problem: str) -> str:
    """Format problem using canonical AIME benchmark prompt."""
    return f"""Solve the following math problem step by step. Put your answer inside \\boxed{{}}.

{problem}

Remember to put your answer inside \\boxed{{}}."""


async def query_scorer_model(
    session: aiohttp.ClientSession,
    problem: str,
    semaphore: asyncio.Semaphore,
) -> str:
    """Query the scorer model to solve a problem."""
    async with semaphore:
        headers = {
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        }

        data = {
            "model": SCORER_MODEL,
            "messages": [
                {"role": "system", "content": AIME_SYSTEM_PROMPT},
                {"role": "user", "content": format_aime_problem(problem)},
            ],
            "temperature": 0,
        }

        async with session.post(OPENROUTER_URL, headers=headers, json=data) as response:
            response.raise_for_status()
            result = await response.json()
            return result["choices"][0]["message"]["content"]


def parse_aime_answer(response: str) -> str | None:
    """Extract the numerical answer from model response."""
    import re
    
    # Try to find \boxed{...} first (canonical format)
    boxed_match = re.search(r'\\boxed\{(\d+)\}', response)
    if boxed_match:
        return boxed_match.group(1)
    
    # Also try **\boxed{...}** (markdown bold)
    boxed_match = re.search(r'\*\*\\boxed\{(\d+)\}\*\*', response)
    if boxed_match:
        return boxed_match.group(1)
    
    # Try <answer>...</answer> tag
    if "<answer>" in response and "</answer>" in response:
        answer = response.split("<answer>")[1].split("</answer>")[0].strip()
        match = re.search(r'\d+', answer)
        if match:
            return match.group()
    
    # Fallback: look for "answer is X" or "= X" patterns at the end
    lines = response.strip().split('\n')
    for line in reversed(lines[-5:]):
        match = re.search(r'(?:answer\s*(?:is|:)\s*|=\s*|\*\*)\s*(\d+)', line, re.IGNORECASE)
        if match:
            return match.group(1)
    
    return None


async def evaluate_aime_sample(
    session: aiohttp.ClientSession,
    sample: dict,
    ground_truth: str,
    semaphore: asyncio.Semaphore,
) -> dict | None:
    """Evaluate a single AIME sample."""
    try:
        response = await query_scorer_model(session, sample['transcript'], semaphore)
        predicted = parse_aime_answer(response)
        
        is_correct = predicted is not None and str(predicted) == str(ground_truth)
        
        return {
            "id": sample['id'],
            "source": sample['source'],
            "ground_truth": ground_truth,
            "predicted": predicted,
            "is_correct": is_correct,
            "model_response": response,
        }
    except Exception as e:
        print(f"Error on {sample['id']}: {e}")
        return {
            'id': sample['id'],
            'source': sample['source'],
            'ground_truth': ground_truth,
            'predicted': 'N/A',
            'is_correct': False,
            'model_response': 'Error: \n{e}'

        }


async def evaluate_aime_benchmark(
    dataset_name: str = "aime",
    max_concurrent: int = 10,
):
    """
    Evaluate model performance on AIME benchmark.
    
    Args:
        dataset_name: "aime" for original, "aime_rephrased" for rephrased version
        max_concurrent: Maximum concurrent API calls
    """
    with open('data/prompt_data.json', 'r') as f:
        data = json.load(f)
    
    samples = data[dataset_name]
    
    # Get ground truth answers (from original aime dataset)
    aime_answers = {s['id']: s['ans'] for s in data['aime']}
    
    # For rephrased, map back to original ID to get answer
    def get_answer(sample):
        sample_id = sample['id']
        if sample_id in aime_answers:
            return aime_answers[sample_id]
        return None
    
    semaphore = asyncio.Semaphore(max_concurrent)
    
    print(f"Evaluating {dataset_name} with {SCORER_MODEL}...")
    print(f"Total samples: {len(samples)}")
    
    async with aiohttp.ClientSession() as session:
        tasks = [
            evaluate_aime_sample(session, sample, get_answer(sample), semaphore)
            for sample in samples
        ]
        results = await tqdm.gather(*tasks)
    
    results = [r for r in results if r is not None]
    
    # Calculate metrics
    correct = sum(1 for r in results if r['is_correct'])
    total = len(results)
    accuracy = correct / total if total > 0 else 0
    
    print(f"\n{'='*50}")
    print(f"Results for {dataset_name}")
    print(f"{'='*50}")
    print(f"Model: {SCORER_MODEL}")
    print(f"Correct: {correct}/{total}")
    print(f"Accuracy: {accuracy:.1%}")
    
    # Save results
    output = {
        "dataset": dataset_name,
        "model": SCORER_MODEL,
        "total_samples": total,
        "correct": correct,
        "accuracy": accuracy,
        "results": results,
    }
    
    output_file = f'results/benchmark_{dataset_name}_{SCORER_MODEL.replace("/", "_")}.json'
    with open(output_file, "w") as f:
        json.dump(output, f, indent=2)
    print(f"Results saved to {output_file}")
    
    return output


async def compare_aime_performance():
    """Compare model performance on original vs rephrased AIME."""
    print("="*60)
    print("AIME Benchmark Performance Comparison")
    print("="*60)
    
    # Evaluate both versions
    original_results = await evaluate_aime_benchmark("aime")
    print()
    rephrased_results = await evaluate_aime_benchmark("aime_rephrased")
    
    # Summary comparison
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    print(f"Model: {SCORER_MODEL}")
    print(f"Original AIME:   {original_results['accuracy']:.1%} ({original_results['correct']}/{original_results['total_samples']})")
    print(f"Rephrased AIME:  {rephrased_results['accuracy']:.1%} ({rephrased_results['correct']}/{rephrased_results['total_samples']})")
    
    diff = rephrased_results['accuracy'] - original_results['accuracy']
    print(f"Difference:      {diff:+.1%}")
    
    return original_results, rephrased_results

async def helper_func_only_rephrased():
    """Compare model performance on original vs rephrased AIME."""
    print("="*60)
    print("AIME Benchmark Performance Comparison")
    print("="*60)
    
    # Evaluate both versions
    original_results_file = f'results/benchmark_aime_{SCORER_MODEL.split("/")[-1]}.json'
    with open(original_results_file, 'r') as f:
        original_results = json.load(f)
    rephrased_results = await evaluate_aime_benchmark("aime_rephrased_v3")
    
    # Summary comparison
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    print(f"Model: {SCORER_MODEL}")
    print(f"Original AIME:   {original_results['accuracy']:.1%} ({original_results['correct']}/{original_results['total_samples']})")
    print(f"Rephrased AIME:  {rephrased_results['accuracy']:.1%} ({rephrased_results['correct']}/{rephrased_results['total_samples']})")
    
    diff = rephrased_results['accuracy'] - original_results['accuracy']
    print(f"Difference:      {diff:+.1%}")
    
    return original_results, rephrased_results

In [ ]:
# Run benchmark comparison
original, rephrased = await helper_func_only_rephrased()

## Benchmark Eval Awareness

In [ ]:
async def query_model(session: aiohttp.ClientSession, prompt: str, semaphore: asyncio.Semaphore) -> str:
    async with semaphore:
        headers = {
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        }

        data = {
            "model": TARGET_MODEL,
            "messages": [
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt},
            ],
            "temperature": 0,
        }

        async with session.post(OPENROUTER_URL, headers=headers, json=data) as response:
            response.raise_for_status()
            result = await response.json()
            return result["choices"][0]["message"]["content"]

def format_prompt(transcript: str, probe_question: str) -> str:
    """Format the transcript with the probe question."""
    return f"""{transcript}\n\n---\n\n{probe_question}"""


def parse_response(content: str, eval_marker: str, real_marker: str) -> tuple[int, str]:
    """Parse model response to extract label and explanation."""
    score = 0
    if "<label>" in content and "</label>" in content:
        label = content.split("<label>")[1].split("</label>")[0].strip().upper()
        score = 1 if eval_marker in label else 0
    elif eval_marker in content.upper() and real_marker not in content.upper():
        score = 1
    elif real_marker in content.upper():
        score = 0
    else:
        eval_pos = content.upper().find("EVAL")
        real_pos = content.upper().find("REAL")
        if eval_pos != -1 and (real_pos == -1 or eval_pos < real_pos):
            score = 1

    explanation = ""
    if "<explanation>" in content and "</explanation>" in content:
        explanation = content.split("<explanation>")[1].split("</explanation>")[0].strip()

    return score, explanation


async def process_sample(
    session: aiohttp.ClientSession,
    sample: dict,
    probe_text: str,
    semaphore: asyncio.Semaphore,
    eval_marker: str,
    real_marker: str,
    debug: bool = False,
) -> dict | None:
    """Process a single sample."""
    if debug:
        print(f'sample: {sample}')

    prompt = format_prompt(sample['transcript'], probe_text)

    try:
        content = await query_model(session, prompt, semaphore)
        score, explanation = parse_response(content, eval_marker, real_marker)

        return {
            "id": sample['id'],
            "source": sample['source'],
            "is_eval": sample['is_eval'],
            "predicted": score,
            "user_prompt": sample['transcript'],
            "explanation": explanation,
        }
    except Exception as e:
        print(f"Error on {sample['id']}: {e}")
        return None


In [ ]:
def calculate_metrics(results: list[dict], eval_dataset_name: str, debug=True):
    y_true = [r["is_eval"] for r in results]
    y_pred = [r["predicted"] for r in results]

    accuracy = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)

    if debug:
        print(f"Total samples: {len(results)}")
        print(f"Accuracy: {accuracy:.3f}")
        print(f"AUC: {auc:.3f}")

    eval_results = [r for r in results if r["source"] == eval_dataset_name]
    sharegpt_results = [r for r in results if r["source"] == "sharegpt"]

    eval_correct = sum(1 for r in eval_results if r["predicted"] == 1)
    sharegpt_correct = sum(1 for r in sharegpt_results if r["predicted"] == 0)

    accuracy_for_eval = eval_correct / len(eval_results)
    accuracy_for_real = sharegpt_correct / len(sharegpt_results)

    if debug:
        print(f"{eval_dataset_name} dataset (should be EVAL): {eval_correct}/{len(eval_results)} correct ({eval_correct/len(eval_results)*100:.1f}%)")
        print(f"ShareGPT (should be REAL): {sharegpt_correct}/{len(sharegpt_results)} correct ({sharegpt_correct/len(sharegpt_results)*100:.1f}%)")

    return accuracy, auc, accuracy_for_eval, accuracy_for_real


async def test_eval_awareness(
    eval_dataset_name: str,
    probe_question_idx: int = 0,
    max_concurrent: int = 10,
    debug: bool = False,
):
    probe_q = PROBE_QUESTIONS[probe_question_idx]

    with open('data/prompt_data.json', 'r') as f:
        data = json.load(f)
        eval_samples = data[eval_dataset_name]
        real_samples = data['sharegpt']
        real_samples = real_samples[:min(len(eval_samples), len(real_samples))]
        all_samples = eval_samples + real_samples
    
    semaphore = asyncio.Semaphore(max_concurrent)
    async with aiohttp.ClientSession() as session:
        tasks = [
            process_sample(
                session, 
                sample, 
                probe_q["text"], 
                semaphore, 
                eval_marker=probe_q['label_eval'], 
                real_marker=probe_q['label_real'], 
                debug=debug,
            )
            for sample in all_samples
        ]
        results = await tqdm.gather(*tasks)

    results = [r for r in results if r is not None]
    accuracy, auc, accuracy_for_eval, accuracy_for_real = calculate_metrics(results, eval_dataset_name)

    output = {
        "probe_question": probe_q,
        "model": TARGET_MODEL,
        "n_samples_per_class": len(eval_samples),
        "accuracy": accuracy,
        "auc": auc,
        "accuracy_for_eval": accuracy_for_eval,
        "accuracy_for_real": accuracy_for_real,
        "results": results,
    }

    output_file = f'results/awareness_{eval_dataset_name}_PQ-{probe_q["id"]}.json'
    with open(output_file, "w") as f:
        json.dump(output, f, indent=2)
    print(f"\nResults saved to {output_file}")

    return output


async def test_eval_awareness_all_pqs(
    eval_dataset_name: str,
    max_concurrent: int = 10,
    debug: bool = False,
):
    """
    Run evaluation awareness test across ALL probe questions and output combined results.
    
    Returns a single JSON file with:
    - Aggregated metrics (mean across all probe questions)
    - Per-probe-question breakdown with individual results
    """
    # Load data once
    with open('data/prompt_data.json', 'r') as f:
        data = json.load(f)
        eval_samples = data[eval_dataset_name]
        real_samples = data['sharegpt']
        real_samples = real_samples[:min(len(eval_samples), len(real_samples))]
        all_samples = eval_samples + real_samples
    
    per_pq_results = {}
    
    # Run evaluation for each probe question
    for pq_idx, probe_q in enumerate(PROBE_QUESTIONS):
        pq_id = probe_q["id"]
        print(f"\n{'='*50}")
        print(f"Running Probe Question {pq_id} ({pq_idx + 1}/{len(PROBE_QUESTIONS)})")
        print(f"{'='*50}")
        
        semaphore = asyncio.Semaphore(max_concurrent)
        async with aiohttp.ClientSession() as session:
            tasks = [
                process_sample(
                    session,
                    sample,
                    probe_q["text"],
                    semaphore,
                    eval_marker=probe_q['label_eval'],
                    real_marker=probe_q['label_real'],
                    debug=debug,
                )
                for sample in all_samples
            ]
            results = await tqdm.gather(*tasks)
        
        results = [r for r in results if r is not None]
        accuracy, auc, accuracy_for_eval, accuracy_for_real = calculate_metrics(
            results, eval_dataset_name, debug=True
        )
        
        per_pq_results[pq_id] = {
            "probe_question": probe_q,
            "accuracy": accuracy,
            "auc": auc,
            "accuracy_for_eval": accuracy_for_eval,
            "accuracy_for_real": accuracy_for_real,
            "results": results,
        }
    
    # Calculate aggregated metrics
    accuracies = [pq["accuracy"] for pq in per_pq_results.values()]
    aucs = [pq["auc"] for pq in per_pq_results.values()]
    eval_accs = [pq["accuracy_for_eval"] for pq in per_pq_results.values()]
    real_accs = [pq["accuracy_for_real"] for pq in per_pq_results.values()]
    
    aggregated = {
        "mean_accuracy": sum(accuracies) / len(accuracies),
        "mean_auc": sum(aucs) / len(aucs),
        "mean_accuracy_for_eval": sum(eval_accs) / len(eval_accs),
        "mean_accuracy_for_real": sum(real_accs) / len(real_accs),
    }
    
    # Print summary
    print(f"\n{'='*60}")
    print("AGGREGATED RESULTS (across all probe questions)")
    print(f"{'='*60}")
    print(f"Model: {TARGET_MODEL}")
    print(f"Dataset: {eval_dataset_name}")
    print(f"Mean Accuracy: {aggregated['mean_accuracy']:.3f}")
    print(f"Mean AUC: {aggregated['mean_auc']:.3f}")
    print(f"Mean Eval Detection: {aggregated['mean_accuracy_for_eval']:.1%}")
    print(f"Mean Real Detection: {aggregated['mean_accuracy_for_real']:.1%}")
    
    # Build output
    output = {
        "eval_dataset": eval_dataset_name,
        "model": TARGET_MODEL,
        "n_samples_per_class": len(eval_samples),
        "n_probe_questions": len(PROBE_QUESTIONS),
        "aggregated": aggregated,
        "per_probe_question": per_pq_results,
    }
    
    output_file = f'results/awareness_{eval_dataset_name}_all_PQs_{TARGET_MODEL.split("/")[-1]}.json'
    with open(output_file, "w") as f:
        json.dump(output, f, indent=2)
    print(f"\nResults saved to {output_file}")
    
    return output

In [ ]:
# Run single probe question
# await test_eval_awareness('aime_rephrased')

# Run ALL probe questions with aggregated results
# await test_eval_awareness_all_pqs('aime')
# await test_eval_awareness_all_pqs('aime_rephrased')
await test_eval_awareness_all_pqs('aime_rephrased_v3')
pass

## Data Viz